# Imports

In [ ]:
# Para melhor organização dos dados em tabelas/matrizes
import pandas as pd

# Para mostrar gráficos
import matplotlib.pyplot as plt

# Matriz de confusão para comparação de resultado
from sklearn.metrics import ConfusionMatrixDisplay

import math
import numpy as np

# Dados

In [ ]:
dados = pd.read_csv('ml-worker/checkpoint_results.csv', sep=',', na_values=["", " ", "NA", "N/A"])
#classes = list(dados.columns).remove('true_predicted')
classes = ['battery', 'glass', 'metal', 'organic', 'paper', 'plastic']
print(classes)

In [ ]:
dados

## Métricas de Avaliação

In [ ]:
metricas = ['accuracy', 'precision', 'recall', 'f1-score']
metricas_avaliacao = pd.DataFrame(index=classes, columns=metricas)

In [ ]:
for classe in classes:
    vp = 0; fp = 0; vn = 0; fn = 0
    i = 0; j = 0
    while i < len(classes):
        j = 0
        while j < len(classes):
            classe1 = classes[i] # Classe prevista
            classe2 = classes[j] # Classe verdadeira
            cont = dados[classe1][j]
            if (classe1 == classe2): # Se previu corretamente
                if (classe == classe1): # Classe é a atual
                    vp += cont
                else: # Classe é outra
                    vn += cont
            else: # Se previu errado
                if (classe == classe1): # Classe prevista é a atual
                    fp += cont
                elif (classe == classe2): # Classe correta é a atual
                    fn += cont
                else: # Não relacionado
                    vn += cont
            j = j + 1
        i = i + 1
    precisao = vp / (vp + fp)
    recall = vp / (vp + fn)
    acuracia = (vp + vn) / (vp + vn + fp + fn)
    f1_score = (2 * recall * precisao) / (recall + precisao)
    metricas_avaliacao.loc[classe, 'accuracy'] = acuracia
    metricas_avaliacao.loc[classe, 'precision'] = precisao
    metricas_avaliacao.loc[classe, 'recall'] = recall
    metricas_avaliacao.loc[classe, 'f1-score'] = f1_score

In [ ]:
metricas_avaliacao

## Matriz de Confusão

Para o método da matriz de confusão, precisamos não da matriz já montada, mas de dois arrays, um com o resultado verdadeiro, e outro com o previsto

In [ ]:
dados_array_true = []
dados_array_predicted = []

i = 0; j = 0
while i < len(classes):
    j = 0
    while j < len(classes):
        classe1 = classes[i] # Classe prevista
        classe2 = classes[j] # Classe verdadeira
        n = dados[classe1][j]
        for k in range(n):
            dados_array_true.append(classe2)
            dados_array_predicted.append(classe1)
        j = j + 1
    i = i + 1

# Resultados

In [ ]:
classification_labels = classes
colormap = 'YlOrRd' # Amarelo a Vermelho
cores_classes = ['orange', 'green', 'yellow', 'brown', 'blue', 'red']
cores_metricas = ['green', 'orange', 'yellow', 'red']

## Métricas de Avaliação 

In [ ]:
width = 0.2
cor = 0
pos = 0 - math.floor(len(metricas)/2)
x = np.arange(len(classes))

fig = plt.figure()
ax = fig.add_subplot()

for metrica in metricas:
    plt.bar(x + (width * pos), metricas_avaliacao[metrica], width, color=cores_metricas[cor])
    cor = (cor + 1) % len(cores_metricas)
    pos += 1
plt.xticks(x, classes)
plt.xlabel("Classes")
plt.ylabel("Medidas")
plt.title("Métricas de Precisão por Classe")

fig.legend(list(metricas_avaliacao.columns))

plt.show()

In [ ]:
width = 0.1
cor = 0
pos = 0 - math.floor(len(classes)/2)
x = np.arange(len(metricas))

fig = plt.figure()
ax = fig.add_subplot()

for classe in classes:
    plt.bar(x + (width * pos), metricas_avaliacao.loc[classe], width, color=cores_classes[cor])
    cor = (cor + 1) % len(cores_classes)
    pos += 1
plt.xticks(x, metricas)
plt.xlabel("Métrica de Precisão")
plt.ylabel("Medidas")
plt.title("Classes por Métrica de Precisão")
plt.figure(figsize=(18,10))

fig.legend(list(classes))

plt.show()

## Matriz de Confusão

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_true=dados_array_true, y_pred=dados_array_predicted,
                                        labels=classification_labels, cmap=colormap)